# 20 Newsgroups: TF-IDF vs MiniLM embeddings vs Jev

This notebook reads the artifacts written by the `jevbench` CLI (`results/`) and renders the comparison inline. It does not re-run the expensive steps by default.

Run the benchmarks first (see `README.md`):

```bash
uv run jevbench all
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from jevbench.config import RESULTS_DIR, PLOTS_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
print("results dir:", RESULTS_DIR)

In [ ]:
metrics_path = RESULTS_DIR / "metrics.json"
payloads = list(json.loads(metrics_path.read_text()).values()) if metrics_path.exists() else []
print("methods found:", [p["method"] for p in payloads])
len(payloads)

In [ ]:
quality = pd.DataFrame([
    {
        "method": p["method"],
        "accuracy": p["metrics"]["accuracy"],
        "macro_f1": p["metrics"]["f1_macro"],
        "weighted_f1": p["metrics"]["f1_weighted"],
        "kappa": p["metrics"]["cohen_kappa"],
    }
    for p in payloads
]).sort_values("accuracy", ascending=False)
display(quality.style.format({"accuracy": "{:.4f}", "macro_f1": "{:.4f}", "weighted_f1": "{:.4f}", "kappa": "{:.4f}"}))

In [ ]:
timing_path = RESULTS_DIR / "timing.csv"
if timing_path.exists():
    timing = pd.read_csv(timing_path)
    timing["train_s"] = timing[[c for c in ["grid_search_seconds", "fit_median", "encode_train_median"] if c in timing]].sum(axis=1)
    timing["predict_s"] = timing[[c for c in ["predict_median", "encode_test_median"] if c in timing]].sum(axis=1)
    display(timing[["method", "train_s", "predict_s", "throughput_docs_per_s"] + [c for c in ["wall_clock_seconds", "cost_usd"] if c in timing]])
else:
    print("no timing.csv yet")

In [ ]:
for name in ["timing_bar.png", "tfidf_heatmap.png"]:
    path = PLOTS_DIR / name
    if path.exists():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path)))

In [ ]:
import glob
for path in sorted(glob.glob(str(PLOTS_DIR / "confusion_*.png"))) + sorted(glob.glob(str(PLOTS_DIR / "calibration_*.png"))):
    display(Markdown(f"**{Path(path).name}**"))
    display(Image(filename=path))

In [ ]:
report_path = RESULTS_DIR / "REPORT.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))